# TCRdist neighbor search benchmark

Benchmark sparse neighbor finding under a more advanced sequence similarity metric,
using SymScan as a pre-filter ahead of the exact TCRdist computation. Expected
runtime ~20 min, dominated by the exhaustive reference algorithm.

Writes `../data/tcrdist_benchmark.csv`; plotted by `pub/tcrdist.ipynb`.

In [ ]:
import time

import numpy as np
import pandas as pd
import pyrepseq as prs
from tcrdist.repertoire import TCRrep
from tcrdist.rep_funcs import compute_pw_sparse_out_of_memory

import benchutils as bu

# TCRdist thresholds at which recovered pairs are tallied
max_tcrdists = np.arange(0, 52, 3)
# independent subsamples drawn from the repertoire, and their size
nsamples = 3
n_sequence = 40_000
# SymScan pre-filter Levenshtein thresholds
max_editss = [1, 2, 3]

In [ ]:
bu.describe_env()

In [ ]:
df = pd.read_csv('../data/emerson_HIP00110.tsv.gz', sep='\t')
df = df[df['amino_acid'].apply(prs.isvalidcdr3)]
df = prs.standardize_dataframe(df, col_mapper={'amino_acid' : 'CDR3B',
                                          'v_family' : 'TRBV',
                                         },
                              suppress_warnings=True)
df = df[df['CDR3B'].apply(prs.isvalidcdr3)]
df.dropna(subset='TRBV', inplace=True)
df.drop_duplicates('CDR3B', inplace=True)
df['CDR3Blen'] = df['CDR3B'].apply(len)
df = df[df['CDR3Blen']>5]
df['TRBV'] = df['TRBV'] + '*01'
df.reset_index(drop=True, inplace=True)

In [ ]:
df

In [ ]:
dfs = [df.sample(n_sequence) for i in range(nsamples)]

In [ ]:
# warm-up numba for benchmarking
d = dfs[0]
prs.nearest_neighbor_tcrdist(d, max_edits=2,
                             max_tcrdist=0);

In [ ]:
rows = []

for max_edits in max_editss:
    for sample, d in enumerate(dfs):
        told = time.time()
        prs_nn = prs.nearest_neighbor_tcrdist(d, max_edits=max_edits,
                                        max_tcrdist=max_tcrdists[-1])
        # tallied inside the timed region, as in the original benchmark
        counts = [int((prs_nn[:, 2]<=dist).sum()) for dist in max_tcrdists]
        runtime_s = time.time()-told
        rows += [{'algorithm': 'symscan', 'max_edits': max_edits, 'sample': sample,
                  'max_tcrdist': int(dist), 'n_neighbors': n, 'runtime_s': runtime_s}
                 for dist, n in zip(max_tcrdists, counts)]

In [ ]:
def convert_df_to_tcrdist_form(df: pd.DataFrame):
    mapper = {
            "TRBV": "v_b_gene",
            "CDR3B": "cdr3_b_aa",
            "rearrangement" : 'cdr3_b_nucseq'}
    df = df.rename(columns=mapper)

    df = df[list(mapper.values())]

    if not "count" in df:
        df["count"] = 1

    return df

In [ ]:
n_cpu = bu.available_cpus()

In [ ]:
for sample, d in enumerate(dfs):
    d_tcrdist = convert_df_to_tcrdist_form(d)
    d_tcrdist.reset_index(drop=True, inplace=True)
    tr = TCRrep(cell_df=d_tcrdist, organism='human', chains=['beta'], compute_distances=False)
    told = time.time()
    nn = compute_pw_sparse_out_of_memory(tr, max_distance=max_tcrdists[-1],
                                         pm_pbar=False, row_size=1000, pm_processes=n_cpu)[0]
    counts = [int((nn.data<dist+1).sum()-len(d)) for dist in max_tcrdists]
    runtime_s = time.time()-told
    # max_edits does not apply to the exhaustive reference, left empty
    rows += [{'algorithm': 'exhaustive', 'max_edits': pd.NA, 'sample': sample,
              'max_tcrdist': int(dist), 'n_neighbors': n, 'runtime_s': runtime_s}
             for dist, n in zip(max_tcrdists, counts)]

In [ ]:
tcrdist_df = pd.DataFrame(rows)
tcrdist_df.to_csv('../data/tcrdist_benchmark.csv')
tcrdist_df.head()